In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
import cv2


In [2]:
class UITViICDataset(Dataset):
    def __init__(self, data_file, img_dir, processor, max_length=50):
        self.df = pd.read_csv(data_file)
        self.img_dir = img_dir
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        image_file = self.df.iloc[index]['image_file']
        caption = self.df.iloc[index]['caption']
        
        image_path = os.path.join(self.img_dir, image_file)
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        encoding = self.processor(
            images=image,
            text=caption,
            padding="max_length",
            truncation=True,
            max_length = self.max_length,
            return_tensors="pt"
        )

        encoding = {k: v.squeeze(0) for k, v in encoding.items()}

        labels = encoding["input_ids"].clone()
        pad_token_id = self.processor.tokenizer.pad_token_id
        labels[labels == pad_token_id] = -100

        encoding["labels"] = labels
        return encoding 


In [3]:
from transformers import BlipProcessor

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")

train_dataset = UITViICDataset(
    data_file="./data/processed_train.csv",
    img_dir="./data/images",
    processor=processor,
    max_length=50
)

dev_dataset = UITViICDataset(
    data_file="./data/processed_dev.csv",
    img_dir="./data/images",
    processor=processor,
    max_length=50
)
test_dataset = UITViICDataset(
    data_file="./data/processed_test.csv",
    img_dir="./data/images",
    processor=processor,
    max_length=50
)

BATCH_SIZE = 2
train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

dev_dataloader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

c:\Users\nviquang\Documents\University\ThirdYear\Ky2\TTCS\Repo\SourceCode\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
batch = next(iter(train_dataloader))
print("Các key trong batch:", batch.keys())
print("Kích thước ma trận ảnh:", batch['pixel_values'].shape)
print("Kích thước ma trận text:", batch['input_ids'].shape)

Các key trong batch: dict_keys(['pixel_values', 'input_ids', 'attention_mask', 'labels'])
Kích thước ma trận ảnh: torch.Size([2, 3, 384, 384])
Kích thước ma trận text: torch.Size([2, 50])


In [6]:
from transformers import BlipForConditionalGeneration

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device {device}")

model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
model.to(device)

Device cuda


loading configuration file config.json from cache at C:\Users\nviquang\.cache\huggingface\hub\models--Salesforce--blip-image-captioning-base\snapshots\82a37760796d32b1411fe092ab5d4e227313294b\config.json
`text_config` is `None`. Initializing the `BlipTextConfig` with default values.
`vision_config` is `None`. initializing the `BlipVisionConfig` with default values.
Model config BlipConfig {
  "architectures": [
    "BlipForConditionalGeneration"
  ],
  "dtype": "float32",
  "image_text_hidden_size": 256,
  "initializer_factor": 1.0,
  "initializer_range": 0.02,
  "label_smoothing": 0.0,
  "logit_scale_init_value": 2.6592,
  "model_type": "blip",
  "projection_dim": 512,
  "text_config": {
    "add_cross_attention": false,
    "attention_probs_dropout_prob": 0.0,
    "bos_token_id": 30522,
    "cross_attention_hidden_size": null,
    "decoder_start_token_id": null,
    "encoder_hidden_size": 768,
    "eos_token_id": 2,
    "finetuning_task": null,
    "hidden_act": "gelu",
    "hidden_d

BlipForConditionalGeneration(
  (vision_model): BlipVisionModel(
    (embeddings): BlipVisionEmbeddings(
      (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (encoder): BlipEncoder(
      (layers): ModuleList(
        (0-11): 12 x BlipEncoderLayer(
          (self_attn): BlipAttention(
            (dropout): Dropout(p=0.0, inplace=False)
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (projection): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): BlipMLP(
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
      )
    )
    (post_layernorm): LayerNorm((768,), eps=1e-0

In [8]:
for param in model.vision_model.parameters():
    param.requires_grad= False

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total param: {total_params}")
print(f"Trainable param: {trainable_params}")
print(f"Trainable rate: {trainable_params/total_params:.2f}")


Total param: 223971644
Trainable param: 137881148
Trainable rate: 0.62


In [9]:
from torch.optim import AdamW

optimizer_grouped_parameters = [
    p for p in model.parameters() if p.requires_grad
]
optimizer = AdamW(optimizer_grouped_parameters, lr=5e-5)


In [10]:
# Test pipeline
train_dataset = UITViICDataset(
    data_file="./data/train_tiny.csv",
    img_dir="./data/images",
    processor=processor,
    max_length=50
)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)
EPOCHS = 10
ACCUMULATION_STEPS = 2
PRINT_EVERY = 2

In [11]:
from tqdm import tqdm

# EPOCHS = 10
# ACCUMULATION_STEPS = 8
# PRINT_EVERY = 50

model.train()
print("Training start...")

for epoch in range(EPOCHS):
    print(f"Epoch: {epoch+1}/{EPOCHS}")
    running_loss = 0.0

    progress_bar = tqdm(enumerate(train_dataloader), total=len(train_dataloader), desc="Training")
    for step, batch in progress_bar:
        pixel_values = batch['pixel_values'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss = loss / ACCUMULATION_STEPS
        loss.backward()
        running_loss += loss.item()*ACCUMULATION_STEPS

        if (step+1) % ACCUMULATION_STEPS==0:
            optimizer.step()
            optimizer.zero_grad()

        if (step+1)% PRINT_EVERY==0:
            avg_loss = running_loss/PRINT_EVERY
            progress_bar.set_postfix({'loss': f"{avg_loss:.4f}"})
            running_loss=0.0

Training start...
Epoch: 1/10


Training: 100%|██████████| 9/9 [00:10<00:00,  1.15s/it, loss=4.4535]


Epoch: 2/10


Training: 100%|██████████| 9/9 [00:09<00:00,  1.10s/it, loss=3.1595]


Epoch: 3/10


...

Training: 100%|██████████| 9/9 [00:09<00:00,  1.08s/it, loss=0.7251]


Epoch: 9/10


Training: 100%|██████████| 9/9 [00:08<00:00,  1.00it/s, loss=0.4424]


Epoch: 10/10


Training: 100%|██████████| 9/9 [00:10<00:00,  1.21s/it, loss=0.3353]
